# Stochastic gradient against the full gradient

A port of `compareNNBatchStochGrad.py`.

The regression of the previous notebook, run twice from **identical initial
weights**: once with the stochastic gradient method, once with the full
gradient. This is the measurement behind the table on the deck's slide 11,
and the reason the unit says that stochastic gradient is what makes training
affordable.

It also turns out to be a good illustration of how easily a timing comparison
can measure two things at once -- see the last section.

**Requires `torch`.** Takes a couple of minutes: the full-gradient run does
$120\,000$ gradient steps.

In [ ]:
import math, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

SEED, HIDDEN, EPOCHS = 42, [64, 64], 400
LR, BATCH, NOISE, SAMPLES = 1e-3, 64, 0.08, 300
DOMAIN = (-1.5, 1.5)
FULL_MULT = 300        # the full-gradient run uses EPOCHS * FULL_MULT steps

## The same problem as before

Same target, same noise, same split, same architecture.

In [ ]:
def target_fn(x):
    return torch.sin(2.0 * math.pi * x) + 0.3 * x

def make_dataset(n_samples=SAMPLES, noise_std=NOISE, domain=DOMAIN):
    xs = torch.from_numpy(np.random.uniform(domain[0], domain[1],
                                            size=(n_samples, 1))).to(torch.float32)
    ys = target_fn(xs).to(torch.float32)
    return TensorDataset(xs, ys + noise_std * torch.randn_like(ys))

class MLP(nn.Module):
    def __init__(self, input_dim=1, hidden_sizes=HIDDEN, output_dim=1):
        super().__init__()
        layers, last = [], input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        layers.append(nn.Linear(last, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def stack(dataset):
    return (torch.stack([t[0] for t in dataset]),
            torch.stack([t[1] for t in dataset]))

seed_all(SEED)
ds = make_dataset()
n_val = max(20, int(0.15 * len(ds)))
train_ds, val_ds = torch.utils.data.random_split(ds, [len(ds) - n_val, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
print(f"{len(train_ds)} training samples, {len(train_loader)} minibatches per epoch")

## The two training loops

**Stochastic gradient.** One parameter update per minibatch, so $4$ updates
per epoch here. Over $400$ epochs that is $1600$ updates, each using $64$ of
the $255$ training points.

**Full gradient.** One parameter update per epoch, each using all $255$
points. The original script runs this for `EPOCHS * 300` $=120\,000$ epochs,
which is what it takes for plain gradient descent to get anywhere near the
stochastic result.

In [ ]:
def train_minibatch(model, loader, epochs=EPOCHS, lr=LR, optimizer_cls=torch.optim.Adam):
    opt = optimizer_cls(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def train_fullbatch(model, dataset, epochs, lr=LR, optimizer_cls=torch.optim.SGD):
    opt = optimizer_cls(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    xs, ys = stack(dataset)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(xs), ys)
        loss.backward(); opt.step()
    return model

def final_loss(model, dataset):
    model.eval()
    xs, ys = stack(dataset)
    with torch.no_grad():
        return nn.MSELoss()(model(xs), ys).item()

## Running both from the same starting point

The comparison is only meaningful if the two runs start from identical
weights, so one model is built and its state dictionary copied into both.

In [ ]:
base = MLP()
sd = base.state_dict()

model_sgd = MLP(); model_sgd.load_state_dict(sd)
model_full = MLP(); model_full.load_state_dict(sd)

t0 = time.perf_counter()
model_sgd = train_minibatch(model_sgd, train_loader, EPOCHS, LR, torch.optim.Adam)
t_sgd = time.perf_counter() - t0
loss_sgd = final_loss(model_sgd, train_ds)
print(f"stochastic : {t_sgd:7.3f} s   train_loss {loss_sgd:.6f}   "
      f"{EPOCHS} epochs = {EPOCHS*len(train_loader)} updates")

t0 = time.perf_counter()
model_full = train_fullbatch(model_full, train_ds, EPOCHS * FULL_MULT, LR, torch.optim.SGD)
t_full = time.perf_counter() - t0
loss_full = final_loss(model_full, train_ds)
print(f"full batch : {t_full:7.3f} s   train_loss {loss_full:.6f}   "
      f"{EPOCHS*FULL_MULT} epochs = {EPOCHS*FULL_MULT} updates")

## Against the deck

The deck's table reads

| | Time | Loss | Training epochs |
|:--|--:|--:|--:|
| Stochastic gradient | $0.871$ s | $0.013107$ | $400$ |
| Full gradient | $16.950$ s | $0.011145$ | $12000$ |

The losses reproduce, and so does the shape of the timing -- the full
gradient takes roughly twenty times as long for a marginally lower loss. The
absolute times depend on the machine.

**The epoch count in that table is a typo.** The script computes
`epochs2 = int(epochs*300)`, which with $400$ epochs is $120\,000$, not
$12\,000$. The cell below settles it: at $12\,000$ epochs plain gradient
descent is nowhere near converged, and only $120\,000$ reproduces the slide's
$0.011145$.

In [ ]:
for mult in (30, FULL_MULT):
    m = MLP(); m.load_state_dict(sd)
    t0 = time.perf_counter()
    m = train_fullbatch(m, train_ds, EPOCHS * mult, LR, torch.optim.SGD)
    dt = time.perf_counter() - t0
    print(f"full batch, {EPOCHS*mult:7d} epochs : {dt:7.3f} s   "
          f"train_loss {final_loss(m, train_ds):.6f}")
print("\nthe deck's figure is 0.011145 -- reproduced only by the 120000-epoch run")

## What the comparison actually measures

Look again at the two calls. The stochastic run uses **Adam**; the
full-gradient run uses **plain SGD**. So the gap above is the product of two
separate effects:

1. minibatch updates against full-batch updates, and
2. an adaptive optimiser against a fixed step size.

They can be separated. The comparison to make is at an **equal number of
parameter updates** -- the stochastic run does $4$ updates per epoch and so
$1600$ in total, and the question is what $1600$ *full-batch* updates with the
same optimiser would achieve.

In [ ]:
xs_tr, ys_tr = stack(train_ds)
n_updates = EPOCHS * len(train_loader)
loss_fn = nn.MSELoss()
rows = []

for steps, lab in ((EPOCHS, "full gradient, same epochs"),
                   (n_updates, "full gradient, same updates")):
    m = MLP(); m.load_state_dict(sd)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    t0 = time.perf_counter()
    for _ in range(steps):
        opt.zero_grad()
        loss_fn(m(xs_tr), ys_tr).backward()
        opt.step()
    rows.append((lab, "Adam", steps, time.perf_counter() - t0, final_loss(m, train_ds)))

print(f"{'method':<34}{'optimiser':<11}{'updates':>9}{'time (s)':>11}{'train loss':>13}")
print("-" * 78)
print(f"{'stochastic (minibatch 64)':<34}{'Adam':<11}"
      f"{n_updates:>9}{t_sgd:>11.3f}{loss_sgd:>13.6f}")
print(f"{'full gradient (as in the script)':<34}{'SGD':<11}"
      f"{EPOCHS*FULL_MULT:>9}{t_full:>11.3f}{loss_full:>13.6f}")
for lab, o, s, t, l in rows:
    print(f"{lab:<34}{o:<11}{s:>9}{t:>11.3f}{l:>13.6f}")
print(f"\nnoise floor (noise variance) = {NOISE**2:.6f}")

Three things fall out of that table, and the first two are not what the
slide suggests.

**At this problem size the full gradient is not the expensive option.** Given
the same optimiser and the same number of updates, the full-batch run takes
about the same time as the stochastic one -- sometimes less. With $255$
training points, one full-batch gradient costs barely more than one minibatch
gradient, because the matrices involved are small either way and the overhead
per step dominates. The twenty-fold gap the deck reports is therefore almost
entirely the **optimiser**, not the batch size: plain gradient descent with a
fixed $\gamma=10^{-3}$ is slow on this problem, and would be slow with
minibatches too.

**The full-batch run with the same number of updates reaches a *lower*
training loss -- and that is not obviously a good thing.** Compare it against
the noise floor printed underneath. The samples carry noise of standard
deviation $0.08$, so a perfect fit to $\mathbb{E}[J\mid x]$ would still leave
a squared error of $0.0064$ against the noisy targets. A training loss *below*
that number means the network has begun fitting the noise. The stochastic run
sits above the floor; the full-batch one dips under it. The gradient noise
that minibatching introduces is acting as a brake here, which is one of the
two standard arguments for stochastic gradient -- the other being the deck's
point that the randomness helps escape local minima.

**The argument for stochastic gradient is about large $N$, and this example is
small.** The cost of one full-batch step grows linearly in the number of
samples while the cost of one minibatch step does not, so somewhere above a
few thousand samples the full gradient stops being an option at all. The
lunar-lander notebooks are the realistic case: there the "dataset" is a replay
buffer of $50\,000$ transitions that is still growing, and a full gradient over
it at every environment step is out of the question.

So the two conclusions worth carrying away:

- the deck's table is right about the losses and the $400$ epochs, and its
  $12\,000$ should read $120\,000$;
- the comparison as written varies the optimiser as well as the batch size, so
  the gap it reports should not be read as the cost of the full gradient.

In [ ]:
m_fair = MLP(); m_fair.load_state_dict(sd)
opt = torch.optim.Adam(m_fair.parameters(), lr=LR)
for _ in range(n_updates):
    opt.zero_grad(); nn.MSELoss()(m_fair(xs_tr), ys_tr).backward(); opt.step()

fig, ax = plt.subplots(figsize=(9, 5))
xs = np.linspace(DOMAIN[0], DOMAIN[1], 400).reshape(-1, 1).astype(np.float32)
with torch.no_grad():
    for m, lab, style in ((model_sgd,  "stochastic (Adam, minibatch)", "r-"),
                          (model_full, f"full gradient (SGD, {EPOCHS*FULL_MULT} updates)", "g-"),
                          (m_fair,     f"full gradient (Adam, {n_updates} updates)", "b-")):
        m.eval()
        ax.plot(xs, m(torch.from_numpy(xs)).numpy(), style, lw=2, label=lab)
data_x, data_y = (t.numpy() for t in stack(ds))
ax.scatter(data_x, data_y, s=14, alpha=0.3, color="0.4", label="train (noisy)")
ax.plot(xs, np.sin(2*np.pi*xs) + 0.3*xs, "k--", lw=1.5, label="ground truth")
ax.set_xlabel("$x$"); ax.set_ylabel("$J$")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

All three fits land on top of each other wherever there is data, which is
the last thing worth noticing: the three runs differ by a factor of a hundred
in the number of gradient steps and by two optimisers, and they agree on the
answer. What separates them is cost, not accuracy.